In [0]:
from pyspark.sql import functions as F

In [0]:
BRONZE = "abfss://bronze@pharmadatalakeuk.dfs.core.windows.net"
SILVER = "abfss://silver@pharmadatalakeuk.dfs.core.windows.net"
mes = (spark.read
       .option("header", "true")
       .csv(f"{BRONZE}/mes/mes_batches.csv"))
mes_dedup = mes.dropDuplicates()
print(f"Rows: {mes.count()}")
print(f"After :{mes_dedup.count()}")
print(mes_dedup.select("batch_id").distinct().count())

Rows: 510
After :500
500


In [0]:
#Trim for spaces in all columns using for loop
mes_trimmed = mes_dedup.select([F.trim(F.col(c)).alias(c) for c in mes_dedup.columns])

In [0]:
#mapping different values in column "deviation_flag" --> Yes or No
# If theere is no value that doesnot count it as No --> stays as null
deviation_flag_clean = (
    F.when(F.upper(F.col("deviation_flag")).isin("Y","YES","TRUE","1"), "Yes")
     .when(F.upper(F.col("deviation_flag")).isin("N","NO","FALSE","0"), "No")
)

mes_standardised = mes_trimmed.withColumn("site", F.initcap("site"))\
                .withColumn("production_line", F.upper("production_line"))\
                .withColumn("product_code", F.upper("product_code"))\
                .withColumn("batch_status", F.initcap(F.lower("batch_status")))\
                .withColumn("deviation_flag", deviation_flag_clean)
display(mes_standardised)

batch_id,site,production_line,product_code,therapy_area,planned_qty,actual_qty,yield_pct,batch_status,deviation_flag,start_time,end_time
BATCH-00010,Wuxi,PET-03,PH_NEU_01,Neurology,750,728.70001220703125,97.16,Released,No,2024-01-06 21:32:00.0000000,2024-01-07 14:31:00.0000000
BATCH-00066,Wuxi,PET-02,PH_INF_02,Infectious Disease,1500,1428,95.2,Completed,No,2024-02-17 23:07:00.0000000,2024-02-18 18:11:00.0000000
BATCH-00080,Singapore,PET-03,PH_ONC_01,Oncology,1000,966.4000244140625,96.64,Released,No,2024-02-23 05:52:00.0000000,2024-02-23 21:00:00.0000000
BATCH-00090,Houston,BL-01,PH_IMM_01,Immunology,1250,1203.5,96.28,Completed,No,2024-03-05 16:38:00.0000000,2024-03-06 05:06:00.0000000
BATCH-00091,Wuxi,PET-03,PH_NEU_02,Neurology,750,718.5999755859375,95.81,Released,No,2024-03-06 11:29:00.0000000,2024-03-07 18:31:00.0000000
BATCH-00108,Singapore,PET-03,PH_IMM_01,Immunology,250,236.39999389648438,94.56,Released,No,2024-03-15 01:58:00.0000000,2024-03-15 10:43:00.0000000
BATCH-00126,Singapore,BL-01,PH_NEU_01,Neurology,1250,1216.699951171875,97.34,Released,No,2024-03-23 06:05:00.0000000,2024-03-24 07:11:00.0000000
BATCH-00136,Dublin,PET-03,PH_CARD_02,Cardiology,1500,1416.0999755859375,94.41,Released,No,2024-03-30 00:20:00.0000000,2024-03-31 09:30:00.0000000
BATCH-00137,Wuxi,BL-01,PH_NEU_02,Neurology,750,713.29998779296875,95.11,Completed,No,2024-03-30 08:15:00.0000000,2024-03-30 17:57:00.0000000
BATCH-00147,Singapore,BL-03,PH_GI_01,Others,1000,955.79998779296875,95.58%,Released,No,2024-04-05 21:25:00.0000000,2024-04-06 16:35:00.0000000


In [0]:
#Casting the columns with proper datatypes


#removing th unwanted % symbol using regex and then using coalesce to add avalues if yield_pct is null. --> this is a calculated filed that is used when yield_pct is null
yield_pct_clean = F.coalesce(
    F.regexp_replace(F.col("yield_pct"), "%", "").try_cast("double"),
    F.round(F.col("actual_qty").try_cast("double") / F.col("planned_qty").try_cast("double") * 100, 2)
)

mes_typed = mes_standardised.withColumn("planned_qty", F.col("planned_qty").try_cast("double"))\
                .withColumn("actual_qty", F.col("actual_qty").try_cast("double"))\
                .withColumn("yield_pct", yield_pct_clean)

mes_typed.printSchema()

root
 |-- batch_id: string (nullable = true)
 |-- site: string (nullable = true)
 |-- production_line: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- therapy_area: string (nullable = true)
 |-- planned_qty: double (nullable = true)
 |-- actual_qty: double (nullable = true)
 |-- yield_pct: double (nullable = true)
 |-- batch_status: string (nullable = true)
 |-- deviation_flag: string (nullable = true)
 |-- start_time: string (nullable = true)
 |-- end_time: string (nullable = true)



In [0]:
print("Yield null but actual present:",
      mes_typed.filter(F.col("yield_pct").isNull() & F.col("actual_qty").isNotNull()).count())

print("yield_pct nulls:", mes_typed.filter(F.col("yield_pct").isNull()).count())
print("actual_qty nulls:", mes_typed.filter(F.col("actual_qty").isNull()).count())

Yield null but actual present: 0
yield_pct nulls: 13
actual_qty nulls: 20


In [0]:
display(
    mes_typed.groupBy("batch_status")
    .agg(
        F.count("*").alias("rows"),
        F.sum(F.col("actual_qty").isNull().cast("int")).alias("actual_null"),
        F.sum(F.col("yield_pct").isNull().cast("int")).alias("yield_null"),
    )
)

batch_status,rows,actual_null,yield_null
Released,349,7,1
Completed,107,1,0
On Hold,27,0,0
In Progress,12,12,12
Rejected,5,0,0


In [0]:
def parse_timestamp(column_name):
    col = F.col(column_name)
    return F.coalesce(
        F.try_to_timestamp(col),
        F.try_to_timestamp(col, F.lit("dd/MM/yyyy HH:mm")),
        F.try_to_timestamp(col, F.lit("dd-MMM-yyyy HH:mm")),
    )

In [0]:
mes_parsed = mes_typed.withColumn("start_time", parse_timestamp("start_time"))\
                .withColumn("end_time", parse_timestamp("end_time"))
mes_parsed.printSchema()

root
 |-- batch_id: string (nullable = true)
 |-- site: string (nullable = true)
 |-- production_line: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- therapy_area: string (nullable = true)
 |-- planned_qty: double (nullable = true)
 |-- actual_qty: double (nullable = true)
 |-- yield_pct: double (nullable = true)
 |-- batch_status: string (nullable = true)
 |-- deviation_flag: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)



In [0]:
F.expr("try_to_timestamp(start_time, 'yyyy-MM-dd HH:mm:ss')")

Column<'try_to_timestamp(start_time, 'yyyy-MM-dd HH:mm:ss')'>

In [0]:
print("Unparsed start_time:", mes_parsed.filter(F.col("start_time").isNull()).count())
print("Unparsed end_time:  ", mes_parsed.filter(F.col("end_time").isNull()).count())

display(mes_parsed.select("batch_id", "batch_status", "start_time", "end_time").limit(10))

Unparsed start_time: 0
Unparsed end_time:   12


batch_id,batch_status,start_time,end_time
BATCH-00010,Released,2024-01-06T21:32:00.000Z,2024-01-07T14:31:00.000Z
BATCH-00066,Completed,2024-02-17T23:07:00.000Z,2024-02-18T18:11:00.000Z
BATCH-00080,Released,2024-02-23T05:52:00.000Z,2024-02-23T21:00:00.000Z
BATCH-00090,Completed,2024-03-05T16:38:00.000Z,2024-03-06T05:06:00.000Z
BATCH-00091,Released,2024-03-06T11:29:00.000Z,2024-03-07T18:31:00.000Z
BATCH-00108,Released,2024-03-15T01:58:00.000Z,2024-03-15T10:43:00.000Z
BATCH-00126,Released,2024-03-23T06:05:00.000Z,2024-03-24T07:11:00.000Z
BATCH-00136,Released,2024-03-30T00:20:00.000Z,2024-03-31T09:30:00.000Z
BATCH-00137,Completed,2024-03-30T08:15:00.000Z,2024-03-30T17:57:00.000Z
BATCH-00147,Released,2024-04-05T21:25:00.000Z,2024-04-06T16:35:00.000Z


In [0]:
mes_final = (mes_parsed\
    .withColumn("batch_duration_hours",
                F.round((F.col("end_time").cast("long") - F.col("start_time").cast("long")) / 3600, 2))
    .withColumn("yield_exceeds_100", F.col("yield_pct") > 100)
    .withColumn("end_before_start", F.col("end_time") < F.col("start_time")))

In [0]:
print("Rows:", mes_final.count())
print("Yield above 100:", mes_final.filter(F.col("yield_exceeds_100")).count())
print("End before start:", mes_final.filter(F.col("end_before_start")).count())

mes_final.printSchema()

Rows: 500
Yield above 100: 5
End before start: 4
root
 |-- batch_id: string (nullable = true)
 |-- site: string (nullable = true)
 |-- production_line: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- therapy_area: string (nullable = true)
 |-- planned_qty: double (nullable = true)
 |-- actual_qty: double (nullable = true)
 |-- yield_pct: double (nullable = true)
 |-- batch_status: string (nullable = true)
 |-- deviation_flag: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- batch_duration_hours: double (nullable = true)
 |-- yield_exceeds_100: boolean (nullable = true)
 |-- end_before_start: boolean (nullable = true)



In [0]:
(mes_final.write
 .format("delta")
 .mode("overwrite")
 .save(f"{SILVER}/mes"))

# Plain Parquet copy for Snowflake. Overwrite removes old files, so only the current version is kept.
(mes_final.write
 .mode("overwrite")
 .parquet(f"{SILVER}/export/mes"))

check = spark.read.format("delta").load(f"{SILVER}/mes")
print("Written and read back:", check.count())

Written and read back: 500


In [0]:
export_check = spark.read.parquet(f"{SILVER}/export/mes")
print("Export read back:", export_check.count())

Export read back: 500


In [0]:
display(mes_final)

batch_id,site,production_line,product_code,therapy_area,planned_qty,actual_qty,yield_pct,batch_status,deviation_flag,start_time,end_time,batch_duration_hours,yield_exceeds_100,end_before_start
BATCH-00010,Wuxi,PET-03,PH_NEU_01,Neurology,750.0,728.7000122070312,97.16,Released,No,2024-01-06T21:32:00.000Z,2024-01-07T14:31:00.000Z,16.98,false,false
BATCH-00066,Wuxi,PET-02,PH_INF_02,Infectious Disease,1500.0,1428.0,95.2,Completed,No,2024-02-17T23:07:00.000Z,2024-02-18T18:11:00.000Z,19.07,false,false
BATCH-00080,Singapore,PET-03,PH_ONC_01,Oncology,1000.0,966.4000244140625,96.64,Released,No,2024-02-23T05:52:00.000Z,2024-02-23T21:00:00.000Z,15.13,false,false
BATCH-00090,Houston,BL-01,PH_IMM_01,Immunology,1250.0,1203.5,96.28,Completed,No,2024-03-05T16:38:00.000Z,2024-03-06T05:06:00.000Z,12.47,false,false
BATCH-00091,Wuxi,PET-03,PH_NEU_02,Neurology,750.0,718.5999755859375,95.81,Released,No,2024-03-06T11:29:00.000Z,2024-03-07T18:31:00.000Z,31.03,false,false
BATCH-00108,Singapore,PET-03,PH_IMM_01,Immunology,250.0,236.39999389648438,94.56,Released,No,2024-03-15T01:58:00.000Z,2024-03-15T10:43:00.000Z,8.75,false,false
BATCH-00126,Singapore,BL-01,PH_NEU_01,Neurology,1250.0,1216.699951171875,97.34,Released,No,2024-03-23T06:05:00.000Z,2024-03-24T07:11:00.000Z,25.1,false,false
BATCH-00136,Dublin,PET-03,PH_CARD_02,Cardiology,1500.0,1416.0999755859375,94.41,Released,No,2024-03-30T00:20:00.000Z,2024-03-31T09:30:00.000Z,33.17,false,false
BATCH-00137,Wuxi,BL-01,PH_NEU_02,Neurology,750.0,713.2999877929688,95.11,Completed,No,2024-03-30T08:15:00.000Z,2024-03-30T17:57:00.000Z,9.7,false,false
BATCH-00147,Singapore,BL-03,PH_GI_01,Others,1000.0,955.7999877929688,95.58,Released,No,2024-04-05T21:25:00.000Z,2024-04-06T16:35:00.000Z,19.17,false,false
